## 1. 멀티턴을 반영한 쿼리 생성기

이전 질문, 이전 답변, 현재 질문을 바탕으로 멀티턴 대화를 생성하는 쿼리 생성기를 작성하세요.  

답변으로 검색어만 반환하면 됩니다.

In [ ]:
# 키는 Colab 왼쪽 🔑(보안 비밀)에 OPENAI_API_KEY 이름으로 저장하세요
from google.colab import userdata
from openai import OpenAI

# OpenAI API 키 (전역 변수로 관리)
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

In [ ]:
client = OpenAI(api_key=OPENAI_API_KEY)

In [ ]:
def generate_search_query(previous_question: str, previous_answer: str, current_question: str) -> list:
    """
    GPT-4를 이용해 멀티턴 대화 기반 검색어를 재구성하는 함수
    :param previous_question: 이전 질문
    :param previous_answer: 이전 답변
    :param current_question: 현재 질문
    :return: 검색어 리스트
    """
    prompt = f"""
    이전 질문: {previous_question}
    이전 답변: {previous_answer}
    현재 질문: {current_question}

    이전 질문과 이전 답변을 바탕으로 현재 질문에 대한 답을 해주시오.
    """

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "이전 질문과 이전 답변을 바탕으로 현재 질문에 대한 답을 해주시오."},
            {"role": "user", "content": prompt}
        ],
    )

    return response.choices[0].message.content.strip()

In [ ]:
# 사용 예시
previous_question = "서울 여행지 추천좀"
previous_answer = "네 추천해드릴게요"
current_question = "전주는?"

search_query = generate_search_query(previous_question, previous_answer, current_question)
print(search_query)

AuthenticationError: Error code: 401 - {'error': {'message': 'Your API key has expired. Create a new API key to continue.', 'type': 'invalid_request_error', 'code': 'expired_secret_key', 'param': None}, 'status': 401}

In [ ]:
# 테스트용 샘플 데이터 (튜플 형태)
sample_data = [
    ("서울 여행지 추천좀", "네 추천해드릴게요", "전주는?"),
    ("부산 맛집 알려줘", "부산에서 유명한 맛집은~", "그럼 제주도는?"),
    ("한옥마을 추천", "전주 한옥마을이 유명합니다", "서울에도 있어?"),
    ("강원도 여행 어디가 좋아?", "설악산과 속초가 좋습니다", "그럼 바다도 추천해줘"),
    ("제주 카페 추천해줘", "오설록 티하우스가 인기 많아요", "한라산 근처는?"),
    ("겨울 여행지 추천", "강원도의 평창이 좋아요", "그럼 따뜻한 곳은?"),
    ("여름 휴가 어디가 좋아?", "제주도가 좋아요", "동해는 어때?"),
    ("서울 명소 알려줘", "경복궁과 남산타워가 있습니다", "야경 좋은 곳은?"),
    ("가족 여행지 추천", "에버랜드가 괜찮아요", "놀이공원 말고 자연도 있나?"),
    ("국내 드라이브 코스 추천", "춘천 가는 길이 좋습니다", "제주에서도 알려줘")
]

# 샘플 데이터 반복 호출
for idx, (prev_q, prev_a, curr_q) in enumerate(sample_data, 1):
    result = generate_search_query(prev_q, prev_a, curr_q)
    print(f"테스트 {idx}: {result}")

테스트 1: 전주 여행지 추천
테스트 2: 제주도 맛집 추천
테스트 3: "서울 한옥마을 추천"
테스트 4: 강원도 속초 바다 추천
테스트 5: 한라산 근처 추천 카페 또는 제주 한라산 근처 카페 추천
테스트 6: 겨울에 따뜻한 여행지 추천
테스트 7: 여름 휴가 제주도 동해 비교
테스트 8: 서울 야경 명소 추천
테스트 9: 가족 여행지 추천: 자연과 함께할 수 있는 놀이공원 외 여행지
테스트 10: 제주 드라이브 코스 추천


## 2. 멀티 쿼리 생성기

주어진 입력으로부터 다수의 질문으로 분리되도록 해보세요.  

['검색어1', '검색어2', '검색어3']으로 분리되어야 합니다.

In [ ]:
def generate_multi_queries(current_query: str) -> list:
    """
    GPT-4를 이용해 멀티 쿼리를 생성하는 함수
    :param current_query: 현재 입력된 하나의 쿼리
    :return: 검색어 리스트
    """
    prompt = f"""
    여기에 어떤 설명을 할까요? 여러분들이 채워보세요.
    사용자가 입력한 문장: {current_query}
    여기에 어떤 설명을 할까요? 여러분들이 채워보세요.
    """

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "여기에 어떤 설명을 할까요? 여러분들이 채워보세요."},
            {"role": "user", "content": prompt}
        ],
    )

    search_queries = response.choices[0].message.content.strip()

    # 검증 로직. ['검색어1', '검색어2', '검색어3'] 정확하게 이 형태로 안 나오면 에러 날 것.
    # return eval(search_queries)  # JSON 배열로 반환
    return search_queries

In [ ]:
current_query = "서울 여행지랑 전주 여행지랑 제주 여행지 알려줘"
search_queries = generate_multi_queries(current_query)

In [ ]:
search_queries

['서울 여행지', '전주 여행지', '제주 여행지']

In [ ]:
search_queries = """["서울 인기 맛집 추천", "부산 현지인 맛집 리스트", "대구 유명 먹거리", "서울 미슐랭 레스토랑", "부산 해산물 맛집"]"""
print(eval(search_queries))

['서울 인기 맛집 추천', '부산 현지인 맛집 리스트', '대구 유명 먹거리', '서울 미슐랭 레스토랑', '부산 해산물 맛집']


In [ ]:
# 멀티 쿼리 테스트 데이터
multi_query_samples = [
    "서울 여행지랑 전주 여행지랑 제주 여행지 알려줘",
    "부산 해수욕장이랑 강원도 계곡 추천해줘",
    "봄에 가기 좋은 여행지랑 가을 단풍 명소 알려줘",
    "유럽 여행지랑 미국 관광 명소 알려줘",
    "서울 맛집이랑 부산 맛집이랑 대구 맛집 추천해줘",
    "겨울 캠핑 장소랑 여름 캠핑 장소 알려줘",
    "아이랑 가기 좋은 곳이랑 부모님 모시고 가기 좋은 곳 알려줘",
    "국내 힐링 여행지랑 해외 힐링 여행지 알려줘",
    "혼자 여행하기 좋은 곳이랑 친구랑 가기 좋은 곳 추천해줘",
    "제주도 드라이브 코스랑 부산 드라이브 코스 알려줘"
]

# 멀티 쿼리 샘플 반복 호출
for idx, query in enumerate(multi_query_samples, 1):
    queries = generate_multi_queries(query)
    print(f"멀티 쿼리 테스트 {idx}: {queries}")

멀티 쿼리 테스트 1: ['서울 여행지', '전주 여행지', '제주 여행지']
멀티 쿼리 테스트 2: ['부산 해수욕장', '강원도 계곡']
멀티 쿼리 테스트 3: ['봄에 가기 좋은 여행지', '가을 단풍 명소']
멀티 쿼리 테스트 4: ['유럽 여행지', '미국 관광 명소']
멀티 쿼리 테스트 5: ['서울 맛집', '부산 맛집', '대구 맛집']
멀티 쿼리 테스트 6: ['겨울 캠핑 장소', '여름 캠핑 장소']
멀티 쿼리 테스트 7: ['아이랑 가기 좋은 곳', '부모님 모시고 가기 좋은 곳']
멀티 쿼리 테스트 8: ['국내 힐링 여행지', '해외 힐링 여행지']
멀티 쿼리 테스트 9: ['혼자 여행하기 좋은 곳', '친구랑 가기 좋은 곳']
멀티 쿼리 테스트 10: ['제주도 드라이브 코스', '부산 드라이브 코스']


## 3. 멀티턴과 멀티쿼리 대응

여러분들은 멀티턴과 멀티 쿼리를 모두 처리 가능한 질문 생성기를 만들 겁니다.  

검색어(번호): 완성된 검색어

형태로 검색어가 나오도록 프롬프트를 만드세요.

In [ ]:
def generate_multi_turn_multi_queries(previous_query: str, previous_answer: str, current_query: str) -> list:
    """
    GPT-4를 이용해 멀티턴 & 멀티 쿼리를 생성하는 함수
    :param previous_query: 이전 입력
    :param previous_answer: 이전 응답
    :param current_query: 현재 입력
    :return: 검색어 리스트
    """
    prompt = f"""
    여기에 어떤 설명을 할까요? 여러분들이 채워보세요.
    이전 입력: {previous_query}
    이전 답변: {previous_answer}
    현재 입력: {current_query}

    여기에 어떤 설명을 할까요? 여러분들이 채워보세요.
    """

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "여기에 어떤 설명을 할까요? 여러분들이 채워보세요."},
            {"role": "user", "content": prompt}
        ],
    )

    search_queries = response.choices[0].message.content.strip()
    return eval(search_queries)

# 사용 예시
previous_query = "서울 여행지 추천"
previous_answer = "서울 여행지를 추천해드릴게요~"
current_query = "전주도 알려주고 제주 관광 상품도 알려주고 서울 맛집도 알려줘"

search_queries = generate_multi_turn_multi_queries(previous_query, previous_answer, current_query)

for idx, query in enumerate(search_queries, 1):
    print(f"검색어{idx}: {query}")

검색어1: 전주 여행지
검색어2: 제주 관광 상품
검색어3: 서울 맛집


In [ ]:
multi_turn_multi_query_samples = [
    ("서울 여행지 추천해줘", "경복궁과 남산이 좋습니다.",
     "그럼 거기서 야경 보기 좋은 곳은 어디고, 거기 아니면 전주나 제주에서 가볼 만한 곳은?"),

    ("부산 맛집 알려줘", "해운대와 서면 쪽이 유명합니다.",
     "그럼 거기서 분위기 좋은 카페는 어디고, 거기 아니더라도 광안리 근처 맛집 추천해줘."),

    ("강원도 해수욕장 추천", "양양과 속초 쪽이 좋습니다.",
     "그럼 거기 근처에서 캠핑 가능한 곳은 어디고, 거기 아니더라도 아이들이 놀기 좋은 곳은?"),

    ("국내 겨울 여행지 추천", "평창과 강릉이 좋습니다.",
     "그럼 거기서 온천 가능한 곳은 어디고, 거기 아니면 남쪽 지방의 따뜻한 여행지는?"),

    ("서울 근교 드라이브 코스", "양평과 남양주가 좋습니다.",
     "그럼 거기 근처 맛집은 어디고, 아니면 제주나 부산에서 드라이브하기 좋은 곳은?"),

    ("제주 관광 명소 추천", "성산일출봉과 오설록이 좋습니다.",
     "그럼 거기 근처에서 가족이 가기 좋은 곳은 어디고, 아니면 동쪽 지역 명소는?"),

    ("가족 여행지 추천", "에버랜드와 롯데월드가 좋습니다.",
     "그럼 거기서 가까운 자연 속 숙소는 어디고, 아니면 강원도나 충청도 지역 추천해줘."),

    ("봄꽃 명소 추천", "경주와 진해가 좋습니다.",
     "그럼 거기서 사진 찍기 좋은 곳은 어디고, 거기 아니더라도 가을 단풍 명소는 어디가 좋을까?"),

    ("역사 여행지 추천", "부여와 경주가 좋습니다.",
     "그럼 거기서 아이와 같이 갈만한 곳은 어디고, 거기 아니더라도 서울 근처 역사 명소는?"),

    ("나 서울 놀러갈거야. 그러니까 서울 맛집 추천해줘", "홍대가 좋습니다.",
     "오 젊음의 거리지. 거기서 분위기 좋은 식당은 어디고, 아니면 거기 아니라도 되니까 디저트가 맛있는 카페는?")
]

# 반복 호출 예시
for idx, (prev_q, prev_a, curr_q) in enumerate(multi_turn_multi_query_samples, 1):
    queries = generate_multi_turn_multi_queries(prev_q, prev_a, curr_q)
    print(f"멀티턴 & 멀티쿼리 테스트 {idx}: {queries}")

멀티턴 & 멀티쿼리 테스트 1: ['서울 야경 보기 좋은 곳', '전주 가볼 만한 곳', '제주 가볼 만한 곳']
멀티턴 & 멀티쿼리 테스트 2: ['부산 분위기 좋은 카페', '광안리 근처 맛집']
멀티턴 & 멀티쿼리 테스트 3: ['양양 근처 캠핑 가능한 곳', '속초 근처 캠핑 가능한 곳', '아이들이 놀기 좋은 곳']
멀티턴 & 멀티쿼리 테스트 4: ['평창 온천 가능한 곳', '강릉 온천 가능한 곳', '남쪽 지방 따뜻한 여행지']
멀티턴 & 멀티쿼리 테스트 5: ['양평 근처 맛집', '남양주 근처 맛집', '제주 드라이브 코스', '부산 드라이브 코스']
멀티턴 & 멀티쿼리 테스트 6: ['성산일출봉과 오설록 근처 가족 여행지', '제주 동쪽 지역 명소']
멀티턴 & 멀티쿼리 테스트 7: ['에버랜드나 롯데월드와 가까운 자연 속 숙소', '강원도 자연 속 숙소', '충청도 자연 속 숙소']
멀티턴 & 멀티쿼리 테스트 8: ['경주 사진 찍기 좋은 곳', '진해 사진 찍기 좋은 곳', '가을 단풍 명소']
멀티턴 & 멀티쿼리 테스트 9: ['부여에서 아이와 함께 갈만한 곳', '경주에서 아이와 함께 갈만한 곳', '서울 근처 역사 명소']
멀티턴 & 멀티쿼리 테스트 10: ['젊음의 거리 분위기 좋은 식당', '디저트 맛있는 카페']


## 4. 도구 매칭

여러분들은 이전 질문, 이전 답변, 현재 질문으로부터 도구를 매칭할 겁니다.  

도구 결과는 "[도구 이름]: [완성된 검색어]" 형태로 나와야 합니다.

In [ ]:
def match_queries_to_tools(tools, previous_query, previous_answer, current_query):
    """
    GPT-4를 이용해 쿼리를 분석하고, 도구들과 매칭하여 검색어를 생성하는 함수
    :param tools: 도구 목록 [{"name": str, "description": str}, ...]
    :param previous_query: 이전 질문
    :param previous_answer: 이전 답변
    :param current_query: 현재 질문
    :return: {도구명: 검색어} 형태의 리스트
    """
    tool_info = "\n".join(
        [f"- {tool['name']}: {tool['description']}" for tool in tools]
    )

    prompt = f"""
    여기에 어떤 설명을 할까요? 여러분들이 채워보세요.

    사용 가능한 도구 목록은 다음과 같아:
    {tool_info}

    이전 질문: "{previous_query}"
    이전 답변: "{previous_answer}"
    현재 질문: "{current_query}"

    여기에 어떤 설명을 할까요? 여러분들이 채워보세요.
    """

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "여기에 어떤 설명을 할까요? 여러분들이 채워보세요."},
            {"role": "user", "content": prompt}
        ],
    )

    matched_results = response.choices[0].message.content.strip().split("\n")
    return [line.strip() for line in matched_results if line.strip()]

In [ ]:
# 도구 정의
tools = [
    {"name": "TourSpotFinder", "description": "관광지나 산책로 등 여행 및 휴식 장소를 추천합니다."},
    {"name": "FoodieFinder", "description": "맛집이나 카페 등 음식과 관련된 장소를 추천합니다."},
    {"name": "NoTool", "description": "매칭될 도구가 없다면 여기에다가 매칭하세요."},
]

# 예시 입력
previous_query = "서울 관광지 추천해줘"
previous_answer = "홍대 어때요?"
current_query = "그럼 근처에 산책하기 좋은 곳이랑 분위기 좋은 식당은 어디고, 거기 아니더라도 디저트가 맛있는 카페는?"

# 함수 호출
result = match_queries_to_tools(tools, previous_query, previous_answer, current_query)

# 결과 출력
for item in result:
    print(item)

NameError: name 'client' is not defined

In [ ]:
# 도구 정의 (각 도메인에 맞게 예시마다 다름)
tool_examples = [
    # 1. 여행
    [
        {"name": "TourSpotFinder", "description": "관광지에 대한 정보를 조회하거나, 관광지나 산책로 등 여행지를 추천합니다."},
        {"name": "FoodieFinder", "description": "맛집이나 카페를 추천합니다."},
        {"name": "NoTool", "description": "매칭될 도구가 없다면 여기에다가 매칭하세요. 정말 연관이 없는 질문일 경우에만 매칭해야 합니다."},
    ],
    # 2. 영화 및 문화
    [
        {"name": "MovieRecommender", "description": "영화나 드라마를 추천합니다."},
        {"name": "CultureEventFinder", "description": "문화 행사 및 전시회를 추천합니다."},
        {"name": "NoTool", "description": "매칭될 도구가 없다면 여기에다가 매칭하세요. 정말 연관이 없는 질문일 경우에만 매칭해야 합니다."},
    ],
    # 3. 쇼핑
    [
        {"name": "FashionShopFinder", "description": "옷과 액세서리 매장을 추천합니다."},
        {"name": "ElectronicsFinder", "description": "전자제품 쇼핑몰을 추천합니다."},
        {"name": "NoTool", "description": "매칭될 도구가 없다면 여기에다가 매칭하세요. 정말 연관이 없는 질문일 경우에만 매칭해야 합니다."},
    ],
    # 4. 스포츠
    [
        {"name": "SportEventFinder", "description": "스포츠 경기 일정을 추천합니다."},
        {"name": "FitnessCenterFinder", "description": "운동 시설을 추천합니다."},
        {"name": "NoTool", "description": "매칭될 도구가 없다면 여기에다가 매칭하세요. 정말 연관이 없는 질문일 경우에만 매칭해야 합니다."},
    ],
    # 5. 교육 및 강좌
    [
        {"name": "OnlineCourseFinder", "description": "온라인 강좌 및 강의를 추천합니다."},
        {"name": "LocalWorkshopFinder", "description": "지역 워크숍과 세미나를 추천합니다."},
        {"name": "NoTool", "description": "매칭될 도구가 없다면 여기에다가 매칭하세요. 정말 연관이 없는 질문일 경우에만 매칭해야 합니다."},
    ],
    # 6. 부동산
    [
        {"name": "HouseFinder", "description": "부동산 매물 및 집을 추천합니다."},
        {"name": "InteriorDesignFinder", "description": "인테리어 관련 업체를 추천합니다."},
        {"name": "NoTool", "description": "매칭될 도구가 없다면 여기에다가 매칭하세요. 정말 연관이 없는 질문일 경우에만 매칭해야 합니다."},
    ],
    # 7. 금융
    [
        {"name": "InvestmentAdvisor", "description": "투자 상품과 관련 정보를 제공합니다."},
        {"name": "BankFinder", "description": "은행 및 금융기관 정보를 추천합니다."},
        {"name": "NoTool", "description": "매칭될 도구가 없다면 여기에다가 매칭하세요. 정말 연관이 없는 질문일 경우에만 매칭해야 합니다."},
    ],
    # 8. 의료 및 건강
    [
        {"name": "HospitalFinder", "description": "병원 및 진료 시설을 추천합니다."},
        {"name": "PharmacyFinder", "description": "약국을 추천합니다."},
        {"name": "NoTool", "description": "매칭될 도구가 없다면 여기에다가 매칭하세요. 정말 연관이 없는 질문일 경우에만 매칭해야 합니다."},
    ],
    # 9. IT 및 기술
    [
        {"name": "TechSupportFinder", "description": "기술 지원 및 수리 업체를 추천합니다."},
        {"name": "SoftwareFinder", "description": "소프트웨어 제품을 추천합니다."},
        {"name": "NoTool", "description": "매칭될 도구가 없다면 여기에다가 매칭하세요. 정말 연관이 없는 질문일 경우에만 매칭해야 합니다."},
    ],
    # 10. 취미 및 여가
    [
        {"name": "HobbyClassFinder", "description": "취미 클래스와 강좌를 추천합니다."},
        {"name": "LeisureActivityFinder", "description": "여가 활동 및 체험을 추천합니다."},
        {"name": "NoTool", "description": "매칭될 도구가 없다면 여기에다가 매칭하세요. 정말 연관이 없는 질문일 경우에만 매칭해야 합니다."},
    ]
]

In [ ]:
# 예시 데이터 (멀티턴 & 멀티쿼리)
test_samples = [
    ("서울 여행 추천해줘", "북촌 한옥마을이 좋습니다.",
     "거기서 근처 카페랑 산책로 추천해줘, 아니면 전주 한옥마을은? 그리고 너 누구야?"),

    ("최근 재밌는 영화 추천", "오펜하이머가 흥행했습니다.",
     "그 감독의 다른 영화랑 내가 사는 곳 근처 문화 행사도 알려줘 그리고 크리스토퍼 놀란에 대한 논란도 궁금하네"),

    ("쇼핑할 옷가게 추천해줘", "강남에 편집샵이 많아요.",
     "거기서 가까운 전자제품 매장도 알려줘, 아니면 홍대 쪽도? 그리고 갤럭시 S25 최신 기능은?"),

    ("이번 주 스포츠 경기 추천해줘", "축구 K리그 경기가 많습니다.",
     "거기 근처 피트니스 센터는 어디고, 야구 일정도 알려줘. 야구 선수 중에 누가 야구 경기 제일 잘하냐?"),

    ("온라인에서 배울 만한 강좌 추천", "코세라와 인프런이 좋습니다.",
     "지역에서 참여 가능한 세미나는 뭐가 있고, 영어 강좌는 어디서 들을 수 있어? 그리고 영어 1타 강사 누구냐?"),

    ("서울에서 살기 좋은 아파트 추천해줘", "강남구와 마포구가 인기입니다.",
     "인테리어 업체는 어디가 좋고, 집 가격 시세는 어디서 확인할까? 그리고 요즘 집값 어떠냐?"),

    ("투자할 만한 금융상품 추천", "ETF나 채권이 안전합니다.",
     "좋아. 그럼 그런거 은행 상담 가능한 곳은 어디고, 요즘 금리 정보는 어디서 볼까? 그리고 엔비디아 창업자 누구냐?"),

    ("가까운 병원 추천해줘", "서울대병원이 가깝습니다.",
     "거기 근처 약국은 어디고, 건강검진 가능한 병원은? 그리고 강남에서 가장 유명한 카페 어디냐?"),

    ("컴퓨터가 고장났는데 어디 갈까?", "용산에 수리 업체가 많습니다.",
     "거기서 추천할 만한 소프트웨어는 뭐가 있고, 인터넷 문제는 어디서 해결하지? 인터넷 기사 추천 해줄래?"),

    ("서울에 취미 활동할 곳 추천해줘", "홍대에 클래스가 많습니다.",
     "거기서 체험 가능한 여가 활동은 뭐가 있고, 다른 지역은? 그리고 동호회 추천해볼래?")
]

# 반복 호출로 테스트
for idx, ((prev_q, prev_a, curr_q), tools) in enumerate(zip(test_samples, tool_examples), 1):
    print(f"\n--- 테스트 {idx} ---")
    matched_queries = match_queries_to_tools(tools, prev_q, prev_a, curr_q)
    for mq in matched_queries:
        print(mq)


--- 테스트 1 ---
TourSpotFinder: 북촌 한옥마을 근처 산책로
FoodieFinder: 북촌 한옥마을 근처 카페
TourSpotFinder: 전주 한옥마을 근처 산책로
FoodieFinder: 전주 한옥마을 근처 카페
NoTool: 너 누구야?

--- 테스트 2 ---
MovieRecommender: 크리스토퍼 놀란 감독의 다른 영화
CultureEventFinder: 오펜하이머 영화 근처의 문화 행사
NoTool: 크리스토퍼 놀란에 대한 논란

--- 테스트 3 ---
ElectronicsFinder: 강남 근처 전자제품 매장
ElectronicsFinder: 홍대 근처 전자제품 매장
NoTool: 갤럭시 S25 최신 기능

--- 테스트 4 ---
FitnessCenterFinder: 축구 K리그 경기장 근처 피트니스 센터
SportEventFinder: 이번 주 야구 경기 일정
NoTool: 야구 선수 중 경기 능력이 뛰어난 선수 정보

--- 테스트 5 ---
LocalWorkshopFinder: 지역에서 참여 가능한 세미나
OnlineCourseFinder: 영어 강좌 추천
NoTool: 영어 1타 강사 추천

--- 테스트 6 ---
InteriorDesignFinder: 서울 인테리어 업체 추천
HouseFinder: 서울 집 가격 시세 확인 방법
NoTool: 요즘 서울 집값 현황

--- 테스트 7 ---
BankFinder: 투자 관련 은행 상담 가능한 곳
InvestmentAdvisor: 현재 금융 상품의 금리 정보
NoTool: 엔비디아 창업자 정보

--- 테스트 8 ---
PharmacyFinder: 서울대병원 근처 약국
HospitalFinder: 서울대병원 근처 건강검진 가능한 병원
NoTool: 강남에서 가장 유명한 카페

--- 테스트 9 ---
SoftwareFinder: 용산에서 추천할 만한 소프트웨어
TechSupportFinder: 인터넷 문제 해결을 위한 용산의 수리 업체
NoTool: 인터넷 기사